# VisionFM Baseline Evaluation
Evaluates VisionFM (ViT-Base/16 pretrained with iBOT on fundus images) on IDRiD and APTOS2019.
No adaptation — raw pretrained weights + prototype classifier head.

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import sys, os; sys.path.insert(0, os.getcwd())
import os, sys
_d = os.getcwd()
PROJECT_ROOT = None
while _d != os.path.dirname(_d):
    if os.path.exists(os.path.join(_d, 'setup.py')):
        PROJECT_ROOT = _d
        break
    _d = os.path.dirname(_d)
if PROJECT_ROOT is None:
    for _p in ['/kaggle/working/medical_CTTA', '/content/medical_CTTA']:
        if os.path.isdir(_p):
            PROJECT_ROOT = _p
            break
if PROJECT_ROOT and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.env import init
init()

from src.config import load_config

In [ ]:
# Download VisionFM weights (cached after first run)
from src.models.visionfm_download import download_visionfm_weights
weights_path = download_visionfm_weights(cache_dir="./cache/visionfm")
print(f"VisionFM weights cached at: {weights_path}")

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from src.config import load_config
from src.models.registry import ModelRegistry
from src.data.registry import DatasetRegistry
from src.evaluation.metrics import quadratic_weighted_kappa, per_class_accuracy, overall_accuracy

def evaluate_model(model, dataset_name, data_dir, config, device):
    """Evaluate model on a dataset."""
    dataset_class = DatasetRegistry.get(dataset_name)
    dataset = dataset_class(
        data_dir=data_dir,
        image_size=config.model.image_size,
        train=False,
        normalize_mean=config.model.normalize_mean,
        normalize_std=config.model.normalize_std,
    )
    loader = DataLoader(dataset, batch_size=16, shuffle=False,
                        num_workers=2, pin_memory=True)
    
    model.backbone.eval()
    model.classifier.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    qwk = quadratic_weighted_kappa(all_labels, all_preds)
    acc = overall_accuracy(all_labels, all_preds)
    per_class = per_class_accuracy(all_labels, all_preds, model.get_num_classes())
    return {"qwk": qwk, "accuracy": acc, "per_class": per_class, "n": len(all_labels)}

In [ ]:
config = load_config("configs/model/visionfm.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Model: {config.model.name}")
print(f"Normalize: mean={config.model.normalize_mean}, std={config.model.normalize_std}")

In [ ]:
from src.models.visionfm import VisionFM

model = VisionFM(
    num_classes=config.model.num_classes,
    freeze_layers=config.model.freeze_layers,
    checkpoint=config.model.checkpoint,
    image_size=config.model.image_size,
    normalize_features=config.model.normalize_features,
)
model.load_weights(weights_path)
model = model.to(device)
print(f"Backbone loaded. Feature dim: {model.model_embed_dim}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Evaluate on IDRiD
print("=" * 60)
print("Evaluating on IDRiD (test set)")
print("=" * 60)
results_idrid = evaluate_model(model, "idrid", "./data/IDRiD/", config, device)
print(f"QWK:   {results_idrid['qwk']:.4f}")
print(f"Acc:   {results_idrid['accuracy']*100:.2f}%")
print(f"Per-class: {results_idrid['per_class']}")
print(f"Samples: {results_idrid['n']}")

In [ ]:
# Evaluate on APTOS2019
print("=" * 60)
print("Evaluating on APTOS2019 (test set)")
print("=" * 60)
results_aptos = evaluate_model(model, "aptos2019", "./data/APTOS2019/", config, device)
print(f"QWK:   {results_aptos['qwk']:.4f}")
print(f"Acc:   {results_aptos['accuracy']*100:.2f}%")
print(f"Per-class: {results_aptos['per_class']}")
print(f"Samples: {results_aptos['n']}")

In [ ]:
# Compare with RETFound baseline (if available)
print("=" * 60)
print("VisionFM Baseline Summary")
print("=" * 60)
print(f"{'Dataset':<12} {'QWK':>8} {'Accuracy':>10}")
print(f"{'IDRiD':<12} {results_idrid['qwk']:>8.4f} {results_idrid['accuracy']*100:>9.2f}%")
print(f"{'APTOS2019':<12} {results_aptos['qwk']:>8.4f} {results_aptos['accuracy']*100:>9.2f}%")
print()
print("Feature dim: 3072 (CLS concat from last 4 blocks)")
print(f"Frozen blocks: {config.model.freeze_layers}/12")
print(f"Backbone params: ~86M (ViT-Base)")